# 16 · Monotonic Stack

A **monotonic stack** is a stack whose elements are always kept in **sorted order** (all increasing or all decreasing from bottom to top). Before pushing a new element, we **pop** every element that would break that order.

It turns many O(n²) "look to the left/right for the next bigger/smaller thing" problems into **O(n)**, because every element is pushed once and popped at most once.

**Topics covered**
1. Next Greater Element (brute force → stack)
2. Next Greater Element in a **circular** array
3. Asteroid collision (stack with collisions)

## 1. Next Greater Element (NGE)

For every element, find the **first larger element to its right**; if none exists, answer `-1`.

```
nums = [19, 4, 2, 3, 6, 5, 11, 3, 10, 12]
ans  = [-1, 6, 3, 6, 11, 11, 12, 10, 12, -1]
```

### Approach A – brute force, O(n²)
For each `i`, scan `j = i+1 … n−1` until something bigger shows up.

In [1]:
nums = [19, 4, 2, 3, 6, 5, 11, 3, 10, 12]

ans = [-1] * len(nums)
for i in range(len(nums)):
    for j in range(i + 1, len(nums)):
        if nums[j] > nums[i]:
            ans[i] = nums[j]
            break                                # the FIRST greater element is enough
print("Brute force:", ans)

Brute force: [-1, 6, 3, 6, 11, 11, 12, 10, 12, -1]


### Approach B – monotonic stack, O(n)

**Scan from right to left** and keep a stack of *candidates* for "the next greater element". The stack always holds values that get **larger as you go down** (the top is the smallest, the closest candidate).

For each `nums[i]` (going right → left):
1. **Pop** every stack element `<= nums[i]`. They are smaller than `nums[i]`, and `nums[i]` is closer to anything on the left, so they can never be the *first* greater element for anything to the left again.
2. If the stack is not empty, its **top** is the next greater element of `nums[i]`; otherwise `-1`.
3. **Push** `nums[i]`: it may be the answer for elements further to the left.

Trace for the start of the scan (right → left):
```
i=9  (12): stack empty        → ans=-1   push 12            stack: [12]
i=8  (10): top 12 > 10        → ans=12   push 10            stack: [12, 10]
i=7  (3) : top 10 > 3         → ans=10   push 3             stack: [12, 10, 3]
i=6  (11): pop 3, pop 10 (≤11); top 12 → ans=12  push 11    stack: [12, 11]
```

**Time:** O(n), because each element is pushed once and popped at most once · **Space:** O(n)

In [2]:
def next_greater(nums):
    n = len(nums)
    ans = [-1] * n
    stack = []                                   # candidates; values strictly decrease from bottom to top
    for i in range(n - 1, -1, -1):               # right → left
        while stack and stack[-1] <= nums[i]:    # these can never be a "first greater" for anything on the left
            stack.pop()
        if stack:
            ans[i] = stack[-1]                   # closest remaining larger value
        stack.append(nums[i])
    return ans

print("Stack      :", next_greater(nums))

# Verify against brute force on random inputs
import random
def brute(nums):
    res = [-1] * len(nums)
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[j] > nums[i]:
                res[i] = nums[j]
                break
    return res

random.seed(1)
for _ in range(500):
    arr = [random.randint(-5, 10) for _ in range(random.randint(0, 12))]
    assert next_greater(arr) == brute(arr)
print("✅ matches brute force on 500 random arrays (including duplicates)")

Stack      : [-1, 6, 3, 6, 11, 11, 12, 10, 12, -1]
✅ matches brute force on 500 random arrays (including duplicates)


## 2. Next Greater Element II: circular array

Now the array is **circular**: after the last element, the search wraps around to the first. Example: `[1, 2, 1]` → `[2, -1, 2]` (the last `1` wraps around and finds `2`).

**Trick:** pretend the array is written **twice** (`nums + nums`) and run the same right-to-left algorithm over `2n − 1 … 0`, using `i % n` to read the value. We only **store answers** for the real positions `i < n`; the second copy is used just to fill the stack with the wrapped-around candidates.

**Time:** O(n) (≈ 2n steps) · **Space:** O(n)

In [3]:
class Solution:
    def next_greater_elements(self, nums):
        n = len(nums)
        result = [-1] * n
        stack = []
        for i in range(2 * n - 1, -1, -1):       # walk the doubled array right → left
            value = nums[i % n]
            while stack and stack[-1] <= value:
                stack.pop()
            if i < n and stack:                  # record answers only for the real array positions
                result[i] = stack[-1]
            stack.append(value)                  # push every time so wrapped candidates are available
        return result

sol = Solution()
print(sol.next_greater_elements([1, 2, 1]))                       # [2, -1, 2]
print(sol.next_greater_elements([5, 4, 3, 2, 1]))                 # [-1, 5, 5, 5, 5]
print(sol.next_greater_elements([19, 4, 2, 3, 6, 5, 11, 3, 10, 12]))

def brute_circular(nums):
    n = len(nums)
    res = [-1] * n
    for i in range(n):
        for step in range(1, n):
            if nums[(i + step) % n] > nums[i]:
                res[i] = nums[(i + step) % n]
                break
    return res

for _ in range(500):
    arr = [random.randint(-5, 10) for _ in range(random.randint(0, 12))]
    assert sol.next_greater_elements(arr) == brute_circular(arr)
print("✅ matches brute force on 500 random arrays")

[2, -1, 2]
[-1, 5, 5, 5, 5]
[-1, 6, 3, 6, 11, 11, 12, 10, 12, 19]
✅ matches brute force on 500 random arrays


## 3. Asteroid collision

An array of integers represents asteroids in a row. The **absolute value** is the size and the **sign** is the direction (`+` moves right, `−` moves left). All move at the same speed. When two meet, the **smaller one explodes**; if equal, **both** explode. Two asteroids moving in the same direction never meet. Return the survivors.

Only a **right-moving asteroid followed by a left-moving one** can collide, and the one that matters is always the **most recent right-mover**, which is what a stack stores.

For each asteroid `x`:
- **`x > 0`** (moving right): nothing to its left can hit it, so **push**.
- **`x < 0`** (moving left): it fights the stack top while the top is a right-mover (`> 0`):
  - top **smaller** than `|x|` → top explodes → **pop** and keep fighting
  - top **equal** to `|x|` → both explode → **pop** and `x` is gone
  - top **larger** than `|x|` → `x` explodes (nothing is pushed)
  - stack empty, or the top is also a left-mover → `x` **survives** → push

**Time:** O(n) · **Space:** O(n)

In [4]:
def asteroid_collision(asteroids):
    stack = []
    for x in asteroids:
        if x > 0:
            stack.append(x)                      # right-moving: no one can hit it from the left
        else:
            # keep destroying smaller right-movers on the stack
            while stack and stack[-1] > 0 and stack[-1] < abs(x):
                stack.pop()
            if stack and stack[-1] == abs(x):    # equal sizes: both explode
                stack.pop()
            elif not stack or stack[-1] < 0:     # nothing left to hit, or only left-movers: x survives
                stack.append(x)
            # else: the top is a bigger right-mover → x explodes, push nothing
    return stack

print(asteroid_collision([4, 7, 1, 1, 1, 2, -3, -1, -7, 17, 15, -18, -19]))   # [-18, -19]
print(asteroid_collision([5, 10, -5]))           # [5, 10]   (-5 hits 10 and explodes)
print(asteroid_collision([8, -8]))               # []        (equal → both explode)
print(asteroid_collision([10, 2, -5]))           # [10]      (-5 destroys 2, then loses to 10)
print(asteroid_collision([-2, -1, 1, 2]))        # [-2, -1, 1, 2]  (never meet)

[-18, -19]
[5, 10]
[]
[10]
[-2, -1, 1, 2]


Let's verify the stack solution against a direct simulation that repeatedly resolves the **leftmost** colliding pair.

In [5]:
def asteroid_simulation(asteroids):
    arr = list(asteroids)
    changed = True
    while changed:
        changed = False
        for i in range(len(arr) - 1):
            if arr[i] > 0 and arr[i + 1] < 0:            # first collision from the left
                a, b = arr[i], abs(arr[i + 1])
                if a > b:
                    del arr[i + 1]
                elif a < b:
                    del arr[i]
                else:
                    del arr[i:i + 2]
                changed = True
                break
    return arr

random.seed(3)
for _ in range(800):
    arr = [random.choice([-1, 1]) * random.randint(1, 8) for _ in range(random.randint(0, 10))]
    assert asteroid_collision(arr) == asteroid_simulation(arr), arr
print("✅ matches simulation on 800 random inputs")

✅ matches simulation on 800 random inputs


---
### ✅ Key takeaways
- **Monotonic stack pattern:** before pushing `x`, pop everything that `x` makes useless. Each element is pushed/popped once → **O(n)**.
- **Next greater to the right** → scan right → left (or scan left → right and resolve elements when they get popped).
- **Circular array** → loop `2n` times and use `i % n`.
- **Asteroids** are not a pure monotonic stack, but the same idea works: the stack holds "survivors so far", and a new element pops the ones it destroys.
- Always compare a clever O(n) solution against a simple brute force on random inputs.